### Begin of the Analysis

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import folium
import os
import seaborn as sns

In [ ]:
# communes_littorales = gpd.read_file("../data/communes_littorales/communes_littorales.shp")
# join_label = ['85166', '85060', '85194']
# #merge and unify the labels
# communes_littorales[communes_littorales.INSEE_COM.isin(join_label)]
# #the geometry of Les Sables d'Olonne is the union of the geometry of the three communes
# communes_littorales.loc[communes_littorales.INSEE_COM=='85194', 'geometry'] = communes_littorales.loc[communes_littorales.INSEE_COM.isin(join_label), 'geometry'].unary_union
# communes_littorales.loc[communes_littorales.INSEE_COM=='85194', 'POPULATION'] = communes_littorales.loc[communes_littorales.INSEE_COM.isin(join_label), 'POPULATION'].sum()
# #remove the rows of the first two communes
# join_label = ['85166', '85060']
# communes_littorales = communes_littorales[~communes_littorales.INSEE_COM.isin(join_label)]

# #save
# communes_littorales.to_file("../data/communes_littorales/communes_littorales.shp", driver='ESRI Shapefile')

#### Load of the databases

In [ ]:
df_final = pd.read_excel('../data/communes_littorales_analysis_metadata.xlsx')
df_final['INSEE_COM'] = df_final['INSEE_COM'].astype(str).str.zfill(5)

path = '../../Data/INSEE/grille_densite_7_niveaux_2024.xlsx'
df_densite = pd.read_excel(path, sheet_name='Grille_Densite', skiprows=4)
df_densite['CODGEO'] = df_densite['CODGEO'].astype(str).str.zfill(5)
df_densite = df_densite[['LIBDENS', 'CODGEO']]

communes_littorales = gpd.read_file("../data/communes_littorales/communes_littorales.shp")
communes_littorales['INSEE_COM'] = communes_littorales['INSEE_COM'].astype(str).str.zfill(5)

exposure_df = pd.read_csv('../data/pop_mean_exposed_200m.csv')
exposure_df = exposure_df.rename(columns={'ind': 'pop_2019', 'ind_65_79': 'pop_65_79', 'ind_80p': 'pop_80p', 'ind_snv': 'nvm',
                                          'ind_exposed': 'pop_exposed', 'ind_65_79_exposed': 'pop_65_79_exposed',
                                          'ind_80p_exposed': 'pop_80p_exposed', 'ind_snv_exposed': 'nvm_exposed'})


In [ ]:
# path = '../../Data/OFGL/ofgl-base-communes-consolidee.csv'
# df_ofgl = pd.DataFrame()
# labels = ['Encours de dette', 'Recettes totales hors emprunts', 'Annuité de la dette']
# #open with chunks with Agrégat.isin(labels) to filter the columns
# for chunk in pd.read_csv(path, chunksize=1000000, sep=';', encoding='utf-8'):
#     chunk = chunk[chunk['Agrégat'].isin(labels)]
#     # chunk = chunk.pivot(index='Code commune', columns='Agrégat', values='Valeur')
#     df_ofgl = pd.concat([df_ofgl, chunk], axis=0)

# df_debt = df_ofgl[['Exercice', 'Nom 2023 EPCI', 'Code Siren 2023 EPCI', 'Commune rurale', 'Commune de montagne', 'Commune touristique', 'Agrégat', 'Montant', 'Population totale', 'Code Insee Collectivité']]
# #pivot and make one column for each label
# df_debt['Code Insee Collectivité'] = df_debt['Code Insee Collectivité'].astype(str).str.zfill(5)

# df_debt = df_debt.pivot_table(index=['Exercice','Code Insee Collectivité', 'Nom 2023 EPCI', 'Code Siren 2023 EPCI', 'Commune rurale', 'Commune de montagne', 'Commune touristique'],
#                                columns='Agrégat', values='Montant', aggfunc='sum').reset_index()
# df_debt['relative_debt'] = df_debt['Encours de dette'] / df_debt['Recettes totales hors emprunts']
# df_debt['relative_annuity'] = df_debt['Annuité de la dette'] / df_debt['Recettes totales hors emprunts']
# df_debt['Code Siren 2023 EPCI'] = df_debt['Code Siren 2023 EPCI'].astype(int).astype(str)
# df_debt = df_debt[df_debt.Exercice == 2021]
# df_debt.to_csv('../data/df_debt.csv', index=False)


In [ ]:
df_debt = pd.read_csv('../data/df_debt.csv')

In [ ]:
lu_changes = pd.read_csv('C:/Users/colin/Downloads/obs_artif_conso_com_2009_2023.csv', sep=',', encoding='latin1')
cols = ['idcom', 'surfcom202', 'epci23', 'naf09art23', 'art09act23', 'art09hab23', 'naf19art20', 'naf20art21', 'naf21art22', 'naf22art23']
#takee the columns that contains the col in cols 
lu_changes.columns = lu_changes.columns.str.split(',').str[0]
lu_changes = lu_changes[cols]
lu_changes['idcom'] = lu_changes['idcom'].astype(str).str.zfill(5)
lu_changes = lu_changes.rename(columns={'idcom': 'INSEE_COM', 'surfcom202': 'surf_com_2023', 'epci23': 'EPCI_2023',
                                         'naf09art23': 'naf09_art23', 'art09act23': 'art09_act23', 'art09hab23': 'art09_hab23',
                                         'naf19art20': 'naf19_art20', 'naf20art21': 'naf20_art21', 'naf21art22': 'naf21_art22',
                                         'naf22art23': 'naf22_art23'})
lu_changes['trend'] = ((lu_changes['naf22_art23'] + lu_changes['naf21_art22'] + lu_changes['naf20_art21'] +
                        lu_changes['naf19_art20'])/4 - lu_changes['naf09_art23'] / 14)>0
lu_changes['share_artif'] = (lu_changes['naf09_art23']/ lu_changes['surf_com_2023']) * 1000
lu_changes['share_activity'] = (lu_changes['art09_act23'] / lu_changes['naf09_art23']) * 100
lu_changes['share_habitation'] = (lu_changes['art09_hab23'] / lu_changes['naf09_art23']) * 100

In [ ]:
comm_2011 = pd.read_csv('../data/circulaire_2011.csv')
comm_2011 = comm_2011.rename(columns={'Code Postal': 'INSEE_COM'})
comm_2011['INSEE_COM'] = comm_2011['INSEE_COM'].astype(str).str.zfill(5)


In [ ]:
df_final[df_final.NOM_COM=='Chaillevette']

In [ ]:
df_final = pd.read_excel('../data/communes_littorales_analysis_metadata.xlsx')
df_final['INSEE_COM'] = df_final['INSEE_COM'].astype(str).str.zfill(5)
df_final = df_final.merge(df_densite, left_on='INSEE_COM', right_on='CODGEO', how='left')
df_final = df_final.merge(exposure_df, on='INSEE_COM', how='left')
df_final = df_final.merge(df_debt[['Code Insee Collectivité', 'relative_debt', 'relative_annuity']], how='left', left_on='INSEE_COM', right_on='Code Insee Collectivité')
df_final = df_final.merge(lu_changes, on='INSEE_COM', how='left')
df_final['circulaire_2011'] = df_final['INSEE_COM'].isin(comm_2011['INSEE_COM'])


In [ ]:
df_final = df_final.drop(columns=['pop_2019_x', 'pop_65_79_x', 'pop_80p_x', 'nvm_x', 'pop_exposed_x',
       'pop_65_79_exposed_x', 'pop_80p_exposed_x', 'nvm_exposed_x'])
df_final = df_final.rename(columns={'pop_2019_y': 'pop_2019', 'pop_65_79_y': 'pop_65_79', 'pop_80p_y': 'pop_80p',
                                    'nvm_y': 'nvm', 'pop_exposed_y': 'pop_exposed', 'pop_65_79_exposed_y': 'pop_65_79_exposed',
                                    'pop_80p_exposed_y': 'pop_80p_exposed', 'nvm_exposed_y': 'nvm_exposed'})

In [ ]:
df_final['share_log_soc'] = df_final['log_soc'] / df_final['men']
df_final['share_log_soc_exposed'] = df_final['log_soc_exposed'] / df_final['log_soc']
df_final['pop_0_17'] = df_final['ind_0_3'] + df_final['ind_4_5'] + df_final['ind_6_10'] + df_final['ind_11_17']
df_final['pop_0_17_exposed'] = df_final['ind_0_3_exposed'] + df_final['ind_4_5_exposed'] + df_final['ind_6_10_exposed'] + df_final['ind_11_17_exposed']
df_final['ind_vieillissante'] =( df_final['pop_65_79'] + df_final['pop_80p'])/df_final['pop_0_17']

df_final['ratio_nvm'] = df_final['nvm_exposed'] / df_final['nvm']
df_final['ratio_pop'] = df_final['pop_exposed'] / df_final['pop_2019']
df_final['ratio_old'] = (df_final['pop_65_79_exposed']+ df_final['pop_80p_exposed']) / (df_final['pop_65_79'] + df_final['pop_80p']) * df_final['pop_2019'] / df_final['pop_exposed']
df_final['ratio_log_soc'] = df_final['log_soc_exposed'] / df_final['log_soc']
df_final['ratio_young'] = df_final['pop_0_17_exposed'] / df_final['pop_0_17'] * df_final['pop_2019'] / df_final['pop_exposed']

In [ ]:
#check the annulation of the pprl kept
### until it has not been done again in create_dataframe.ipynb

gaspar_pprn = pd.read_csv('../../Data/GASPAR/gaspar/pprn_gaspar.csv.csv', sep=';')

label = 'Recul du trait de côte et de falaises'
gaspar_ero = gaspar_pprn[gaspar_pprn.lib_risque.str.contains(label, case=False, na=False)]
gaspar_ero['INSEE_COM'] = gaspar_ero['cod_commune'].astype(str).str.zfill(5)
#keep only the last pprn by dat_approbation
gaspar_ero = gaspar_ero.sort_values(by='dat_approbation').drop_duplicates(subset=['INSEE_COM'], keep='last')
#if dat_annulation is not null, set pprl_ero to False
print(f"Number of communes with PPRL Erosion Cancelled: {gaspar_ero['dat_annulation'].notnull().sum()}")
gaspar_ero['pprl_ero'] = gaspar_ero['dat_annulation'].isnull()

label = 'Par submersion marine'
gaspar_sub = gaspar_pprn[gaspar_pprn.lib_risque.str.contains(label, case=False, na=False)]
gaspar_sub['INSEE_COM'] = gaspar_sub['cod_commune'].astype(str).str.zfill(5)
#keep only the last pprn by dat_approbation
gaspar_sub = gaspar_sub.sort_values(by='dat_approbation').drop_duplicates(subset=['INSEE_COM'], keep='last')
#if dat_annulation is not null, set pprl_sub to False
print(f"Number of communes with PPRL Submersion Cancelled: {gaspar_sub['dat_annulation'].notnull().sum()}")
gaspar_sub['pprl_sub'] = gaspar_sub['dat_annulation'].isnull()

communes_littorales = gpd.read_file("../data/communes_littorales/communes_littorales.shp")
communes_littorales = communes_littorales.merge(gaspar_ero[['INSEE_COM', 'pprl_ero']], on='INSEE_COM', how='left')
communes_littorales = communes_littorales.merge(gaspar_sub[['INSEE_COM', 'pprl_sub']], on='INSEE_COM', how='left')
communes_littorales['pprl_ero'] = communes_littorales['pprl_ero'].fillna(False)
communes_littorales['pprl_sub'] = communes_littorales['pprl_sub'].fillna(False)

#### Begin of the analysis

In [ ]:
#print the number of commune with decret_2024, with pprl_ero and pprl and with one of the combinations
print(f"Number of communes with decret_2024: {df_final['decret_2024'].sum()}")
print(f"Number of communes with pprl_ero: {df_final['pprl_ero'].sum()}")
print(f"Number of communes with pprl: {df_final['pprl'].sum()}")
print(f"Number of communes with decret_2024 and pprl_ero: {(df_final['decret_2024'] * df_final['pprl_ero']).sum()}")
print(f"Number of communes with decret_2024 and pprl: {(df_final['decret_2024'] * df_final['pprl']).sum()}")

print(f"Number of communes with decret_2024 and pprl: {(df_final['decret_2024'] * (1-df_final['pprl'])).sum()}")

In [ ]:
#make a plt.bar figure with the number of communes for each departement
c1 = '#ccff33'
c2 = '#70e000'
c3 = '#00b300'
c4 = '#d00000'
c5 = 'grey'
comm_decret_2022 = pd.read_csv("../data/communes_littorales/comm_decret_2022.txt", header=None).squeeze().tolist()
comm_decret_2022 = [str(x).zfill(5) for x in comm_decret_2022]  # Ensure all codes are 5 digits
comm_decret_2023 = pd.read_csv("../data/communes_littorales/comm_decret_2023.txt", header=None).squeeze().tolist()
comm_decret_2024 = pd.read_csv("../data/communes_littorales/comm_decret_2024.txt", header=None).squeeze().tolist()
comm_2022_not_2023 = communes_littorales[communes_littorales['INSEE_COM'].isin(set(comm_decret_2022) - set(comm_decret_2023))]
# Communes in 2023 but not in 2022
comm_2023_not_20224= communes_littorales[communes_littorales['INSEE_COM'].isin(set(comm_decret_2023) - set(comm_decret_2024))]
# Communes in 2024 but not in 2023

comm_off = pd.concat([comm_2022_not_2023, comm_2023_not_20224]).drop_duplicates()

communes_littorales = gpd.read_file("../data/communes_littorales/communes_littorales.shp")

communes_littorales['color'] = np.where(communes_littorales['INSEE_COM'].isin(comm_off.INSEE_COM), c4,
    np.where(communes_littorales['INSEE_COM'].isin(comm_decret_2022), c1,
    np.where(communes_littorales['INSEE_COM'].isin(comm_decret_2023), c2,
             np.where(communes_littorales['INSEE_COM'].isin(comm_decret_2024), c3, c5))))

communes_littorales['departement'] = communes_littorales['INSEE_COM'].str[:2]

com_dep = communes_littorales.groupby(['departement', 'color']).size().reset_index(name='count')
com_dep['share'] = com_dep['count'] / com_dep.groupby('departement')['count'].transform('sum')
# Keep only rows with color in [c1, c2, c3]
com_dep = com_dep[com_dep['color'].isin([c1, c2, c3])]
com_dep['total_in'] = com_dep.groupby('departement')['share'].transform('sum')
#order by total_in
com_dep = com_dep.sort_values(by='total_in', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
# Get the unique colors present in the data, in the order of c1, c2, c3
color_order = ['#00b300', '#70e000', '#ccff33']
com_dep.loc[com_dep['color'] == c1, 'color'] = 'Joined in 2022'
com_dep.loc[com_dep['color'] == c2, 'color'] = 'Joined in 2023'
com_dep.loc[com_dep['color'] == c3, 'color'] = 'Joined in 2024'
com_dep['Year Joined'] = com_dep['color']

# Pivot and reindex to match the order in com_dep
departement_order = com_dep['departement'].unique()
com_dep_pivot = com_dep.pivot(index='departement', columns='Year Joined', values='share').reindex(departement_order)
com_dep_pivot.plot(
    kind='barh', stacked=True, ax=ax, color=color_order
)
ax.set_xlabel('Share of communes of the department in the decree')
ax.set_ylabel('Department')

In [ ]:
m = folium.Map(location=[46.5, 2], zoom_start=5, tiles='OpenStreetMap')

def style_function(feature):
    color = feature['properties']['color']
    return {
        'fillColor': color,
        'color': color,
        'weight': 1,
        'fillOpacity': 0.5
    }

def highlight_function(feature):
    return {
        'weight': 3,
        'color': '#666',
        'fillOpacity': 0.7
    }

# Add the GeoDataFrame to the map with the style function and popup showing the commune name
folium.GeoJson(
    communes_littorales,
    style_function=style_function,
    highlight_function=highlight_function,
    tooltip=folium.GeoJsonTooltip(fields=['NOM_COM'], aliases=['Commune:'])
).add_to(m)

# Add ADM1 (département) borders


#add a legend to explain what are the colors
# m.add_child(folium.LayerControl())
# Add a legend to the map
legend_html = f"""
<div style="position: fixed; 
    bottom: 50px; left: 50px; width: 270px; height: 140px; 
    background-color: white; z-index:9999; 
    font-size:14px; padding: 10px;">
    <b>Legend</b><br>
    <i style="display: inline-block; width: 14px; height: 14px; background: {c1}; margin-right: 5px; border:1px solid #888;"></i> In the decree since 2022<br>
    <i style="display: inline-block; width: 14px; height: 14px; background: {c2}; margin-right: 5px; border:1px solid #888;"></i> Joined in 2023<br>
    <i style="display: inline-block; width: 14px; height: 14px; background: {c3}; margin-right: 5px; border:1px solid #888;"></i> Joined in 2024<br>
    <i style="display: inline-block; width: 14px; height: 14px; background: {c4}; margin-right: 5px; border:1px solid #888;"></i> Once in the decree but left<br>
    <i style="display: inline-block; width: 14px; height: 14px; background: {c5}; margin-right: 5px; border:1px solid #888;"></i> Not in the decree<br>
</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))

# Add a title to the map
title_html = """
<div style="position: fixed; 
    top: 10px; left: 50%; transform: translateX(-50%); 
    background-color: white; z-index:9999; 
    font-size:20px; text-align: center; font-weight: bold; 
    padding: 5px; box-shadow: 0px 4px 6px rgba(0, 0, 0, 0.1);">
    Communes littorales in the decree on coastal erosion
</div>
"""

m.get_root().html.add_child(folium.Element(title_html))

# Add a scale bar to the map
m.add_child(folium.LatLngPopup())

# Display the map
m

#### Comparison based on LIBDENS

In [ ]:
# communes_littorales = gpd.read_file("../data/communes_littorales/communes_littorales.shp")
communes_littorales['INSEE_COM'] = communes_littorales['INSEE_COM'].astype(str).str.zfill(5)
# Choose 7 colors for th 7 LIBDENS categories
labels_libdens = df_final.groupby('LIBDENS')['pop_2019'].sum().sort_values(ascending=False).index.tolist()
colors = sns.color_palette("Set1", n_colors=len(labels_libdens))
libdens_colors = dict(zip(labels_libdens, colors))

# Merge communes_littorales with df_final to get LIBDENS
communes_littorales = communes_littorales.merge(
    df_final[['INSEE_COM', 'LIBDENS']], on='INSEE_COM', how='left'
)

# Assign color based on LIBDENS, convert RGB to hex for folium
def rgb_to_hex(rgb):
    return '#%02x%02x%02x' % tuple(int(255*x) for x in rgb)

communes_littorales['libdens_color'] = communes_littorales['LIBDENS'].map(
    lambda x: rgb_to_hex(libdens_colors[x]) if pd.notnull(x) and x in libdens_colors else "#cccccc"
)

# Create folium map
m = folium.Map(location=[46.5, 2], zoom_start=5, tiles='OpenStreetMap')

def style_function(feature):
    color = feature['properties']['libdens_color']
    return {
        'fillColor': color,
        'color': color,
        'weight': 1,
        'fillOpacity': 0.6
    }

folium.GeoJson(
    communes_littorales,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=['NOM_COM', 'LIBDENS'], aliases=['Commune:', 'Densité:'])
).add_to(m)

# Build legend dynamically from labels_libdens and libdens_colors
legend_items = ""
for label in labels_libdens:
    color = rgb_to_hex(libdens_colors[label])
    legend_items += f'<i style="display:inline-block;width:14px;height:14px;background:{color};margin-right:5px;border:1px solid #888;"></i> {label}<br>'

legend_html = f"""
<div style="position: fixed; bottom: 50px; left: 50px; width: 270px; background-color: white; z-index:9999; font-size:14px; padding: 10px;">
    <b>Légende - Densité (LIBDENS)</b><br>
    {legend_items}
    <i style="display:inline-block;width:14px;height:14px;background:#cccccc;margin-right:5px;border:1px solid #888;"></i> Non renseigné<br>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# Title
title_html = """
<div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); background-color: white; z-index:9999; font-size:20px; text-align: center; font-weight: bold; padding: 5px; box-shadow: 0px 4px 6px rgba(0, 0, 0, 0.1);">
    Communes littorales par densité (LIBDENS)
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))

m.add_child(folium.LatLngPopup())
m


In [ ]:
cols_insee = ['pop_2019', 'ratio_nvm', 'ratio_pop', 'ratio_old', 'ratio_log_soc', 'ratio_young',
'ind_vieillissante','decret_2024']
df_final.groupby('LIBDENS')[cols_insee].mean().sort_values(by='pop_2019', ascending=False)

In [ ]:
cols_lu = ['surf_com_2023', 'naf09_art23', 'share_activity', 'share_habitation', 
           'trend', 'share_artif', 'decret_2024']
df_final.groupby('LIBDENS')[cols_lu].mean().sort_values(by='surf_com_2023', ascending=False)

In [ ]:
cols_geo = ['tx_artif', 'inec', 'nb_ouvrages_erosion',
       'nb_ouvrages_submersion', 'share_superficy_phma', 'taux_catnat_recon',
       'Reconnue', 'pprl', 'pprl_ero', 'share_artif','decret_2024']
df_final.groupby('LIBDENS')[cols_geo].mean().sort_values(by='tx_artif', ascending=False)

In [ ]:
cols_debt = ['relative_debt', 'relative_annuity', 'decret_2024']
df_final.groupby('LIBDENS')[cols_debt].mean().sort_values(by='relative_debt', ascending=False)

#### PPRL Ero - PPRL Submersion - Decret

In [ ]:
# Plot with 8 colors for all combinations of pprl_ero, pprl_sub, decret_2024
def get_color(row):
    # All three
    if row['pprl_ero'] and row['pprl_sub'] and row['decret_2024']:
        return "#1f78b4"  # Blue
    # Ero + Sub
    if row['pprl_ero'] and row['pprl_sub']:
        return "#2ca02c"  # Green
    # Ero + decret
    if row['pprl_ero'] and row['decret_2024']:
        return "#e377c2"  # Pink
    # Sub + decret
    if row['pprl_sub'] and row['decret_2024']:
        return "#ff7f0e"  # Orange
    # Only decret
    if row['decret_2024']:
        return "#d62728"  # Red
    # Only ero
    if row['pprl_ero']:
        return "#9467bd"  # Purple
    # Only sub
    if row['pprl_sub']:
        return "#bcbd22"  # Olive
    # None
    return "#7f7f7f"      # Gray

# Merge decret_2024 if not already present
if 'decret_2024' not in communes_littorales.columns:
    communes_littorales = communes_littorales.merge(
        df_final[['INSEE_COM', 'decret_2024']], on='INSEE_COM', how='left'
    )

communes_littorales['combo_color'] = communes_littorales.apply(get_color, axis=1)

m = folium.Map(location=[46.5, 2], zoom_start=5, tiles='OpenStreetMap')

def style_function(feature):
    color = feature['properties']['combo_color']
    return {
        'fillColor': color,
        'color': color,
        'weight': 1,
        'fillOpacity': 0.6
    }

folium.GeoJson(
    communes_littorales,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=['NOM_COM', 'pprl_ero', 'pprl_sub', 'decret_2024'],
        aliases=['Commune:', 'PPRL Erosion:', 'PPRL Submersion marine:', 'Décret 2024:']
    )
).add_to(m)

# Legend for all 8 combinations
legend_html = """
<div style="position: fixed; bottom: 50px; left: 50px; width: 400px; background-color: white; z-index:9999; font-size:14px; padding: 10px;">
    <b>Légende - PPRL Erosion, Submersion marine, Décret 2024</b><br>
    <i style="display:inline-block;width:14px;height:14px;background:#1f78b4;margin-right:5px;border:1px solid #888;"></i> Erosion + Submersion + Décret 2024<br>
    <i style="display:inline-block;width:14px;height:14px;background:#2ca02c;margin-right:5px;border:1px solid #888;"></i> Erosion + Submersion<br>
    <i style="display:inline-block;width:14px;height:14px;background:#e377c2;margin-right:5px;border:1px solid #888;"></i> Erosion + Décret 2024<br>
    <i style="display:inline-block;width:14px;height:14px;background:#ff7f0e;margin-right:5px;border:1px solid #888;"></i> Submersion + Décret 2024<br>
    <i style="display:inline-block;width:14px;height:14px;background:#d62728;margin-right:5px;border:1px solid #888;"></i> Décret 2024 seulement<br>
    <i style="display:inline-block;width:14px;height:14px;background:#9467bd;margin-right:5px;border:1px solid #888;"></i> Erosion seulement<br>
    <i style="display:inline-block;width:14px;height:14px;background:#bcbd22;margin-right:5px;border:1px solid #888;"></i> Submersion seulement<br>
    <i style="display:inline-block;width:14px;height:14px;background:#7f7f7f;margin-right:5px;border:1px solid #888;"></i> Aucun des trois<br>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

title_html = """
<div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); background-color: white; z-index:9999; font-size:20px; text-align: center; font-weight: bold; padding: 5px; box-shadow: 0px 4px 6px rgba(0, 0, 0, 0.1);">
    Communes littorales - PPRL Erosion, Submersion marine, Décret 2024
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))
m.add_child(folium.LatLngPopup())
m

In [ ]:
#check the annulation of the pprl kept
### until it has not been done again in create_dataframe.ipynb

gaspar_pprn = pd.read_csv('../../Data/GASPAR/gaspar/pprn_gaspar.csv.csv', sep=';')

label = 'Recul du trait de côte et de falaises'
gaspar_ero = gaspar_pprn[gaspar_pprn.lib_risque.str.contains(label, case=False, na=False)]
gaspar_ero['INSEE_COM'] = gaspar_ero['cod_commune'].astype(str).str.zfill(5)
#keep only the last pprn by dat_approbation
gaspar_ero = gaspar_ero.sort_values(by='dat_approbation').drop_duplicates(subset=['INSEE_COM'], keep='last')
#if dat_annulation is not null, set pprl_ero to False
print(f"Number of communes with PPRL Erosion Cancelled: {gaspar_ero['dat_annulation'].notnull().sum()}")
gaspar_ero['pprl_ero'] = gaspar_ero['dat_annulation'].isnull()

label = 'Par submersion marine'
gaspar_sub = gaspar_pprn[gaspar_pprn.lib_risque.str.contains(label, case=False, na=False)]
gaspar_sub['INSEE_COM'] = gaspar_sub['cod_commune'].astype(str).str.zfill(5)
#keep only the last pprn by dat_approbation
gaspar_sub = gaspar_sub.sort_values(by='dat_approbation').drop_duplicates(subset=['INSEE_COM'], keep='last')
#if dat_annulation is not null, set pprl_sub to False
print(f"Number of communes with PPRL Submersion Cancelled: {gaspar_sub['dat_annulation'].notnull().sum()}")
gaspar_sub['pprl_sub'] = gaspar_sub['dat_annulation'].isnull()



#### Load Geographical Maps

In [ ]:
artif_lit = gpd.read_file("../data/N_artificialisation_ouvrage_metropole_epsg2154_L_042018_shape/N_artificialisation_ouvrage_metropole_epsg2154_L.shp")
artif_communes = gpd.overlay(artif_lit, communes_littorales, how='intersection', keep_geom_type=True)
artif_communes = artif_communes[['INSEE_COM', 'l_tx_arti', 'l_emprise', 'decret_2024']].groupby('INSEE_COM').agg({'l_tx_arti': 'sum', 'l_emprise': 'sum', 'decret_2024':'mean'}).reset_index()
artif_communes['tx_artif'] = artif_communes['l_tx_arti'] / artif_communes['l_emprise']





###### Nature of the coastlines

In [ ]:
nature_coastlines = gpd.read_file("../../Data/Cerema/Erosion/N_nature_cote_L_metropole_epsg2154_042019_shape/N_nature_cote_L_metropole_epsg2154.shp")

dict_nature = {
    'Falaise et côte rocheuse supérieure à 20m': 10,
    'Falaise et côte rocheuse inférieure à 20m': 20,
    'Côte d\'accumulation sableuse ou sablo-limoneuse': 30,
    'Côte d\'accumulation sableuse ou sablo-limoneuse': 31,
    'Côte d\'accumulation vaseuse': 40,
    'Côte d\'accumulation vaseuse': 41,
    'Côte artificialisée': 50,
    'Embouchures': 60,
}
#put together the same nature coastlines
nature_coastlines.loc[nature_coastlines.geom_macro==31, 'geom_macro'] = 30
nature_coastlines.loc[nature_coastlines.geom_macro==41, 'geom_macro'] = 40

# Assign a color to each unique 'geom_macro'
cat_geol_unique = np.sort(nature_coastlines['geom_macro'].unique())
color_map = {cat: mcolors.to_hex(plt.colormaps['tab20'](i / len(cat_geol_unique))) for i, cat in enumerate(cat_geol_unique)}
label_map = {v: k for k, v in dict_nature.items()}

def style_function(feature):
    cat = feature['properties']['geom_macro']
    return {
        'color': color_map.get(cat, '#333333'),
        'weight': 3,
        'fillOpacity': 0.7
    }

def tooltip_label(feature):
    cat = feature['properties']['geom_macro']
    return label_map.get(cat, str(cat))

m = folium.Map(location=[46.5, 2], zoom_start=5)
folium.GeoJson(
    nature_coastlines,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=['geom_macro'],
        aliases=['Type:'],
        labels=False,
        sticky=True,
        parse_html=True,
        localize=False
    ),
    highlight_function=lambda x: {'weight': 5, 'color': '#666'}
).add_to(m)

title_html = """
<div style="position: fixed; 
    top: 10px; left: 50%; transform: translateX(-50%); 
    background-color: white; z-index:9999; 
    font-size:20px; text-align: center; font-weight: bold; 
    padding: 5px; box-shadow: 0px 4px 6px rgba(0, 0, 0, 0.1);">
    Nature of Coastlines by Category
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))

m


In [ ]:
nature_coastlines = gpd.read_file("../../Data/Cerema/Erosion/N_nature_cote_L_metropole_epsg2154_042019_shape/N_nature_cote_L_metropole_epsg2154.shp")
nature_coastlines.loc[nature_coastlines.geom_macro==31, 'geom_macro'] = 30
nature_coastlines.loc[nature_coastlines.geom_macro==41, 'geom_macro'] = 40
nature_coastlines = gpd.overlay(nature_coastlines, communes_littorales, how='intersection', keep_geom_type=True)
nature_coastlines = nature_coastlines[['geom_macro', 'long', 'INSEE_COM']].groupby(['INSEE_COM', 'geom_macro']).agg({'long': 'sum'}).reset_index()
nature_coastlines = nature_coastlines.pivot(index='INSEE_COM', columns='geom_macro', values='long').reset_index()
dict_nature = {
    'Falaise et côte rocheuse supérieure à 20m': 10,
    'Falaise et côte rocheuse inférieure à 20m': 20,
    'Côte d\'accumulation sableuse ou sablo-limoneuse': 30,
    'Côte d\'accumulation vaseuse': 40,
    'Côte artificialisée': 50,
    'Embouchures': 60,
}
# Inverse mapping: replace numeric columns with their string labels from dict_nature
inv_dict_nature = {v: k for k, v in dict_nature.items()}
nature_coastlines.columns = ['INSEE_COM'] + [inv_dict_nature.get(col, col) for col in nature_coastlines.columns[1:]]
nature_coastlines['Total'] = nature_coastlines.iloc[:, 1:].sum(axis=1)

In [ ]:
share_nat_coastlines = nature_coastlines.copy()
share_nat_coastlines = share_nat_coastlines.fillna(0)
share_nat_coastlines.iloc[:, 1:-1] = share_nat_coastlines.iloc[:, 1:-1].div(share_nat_coastlines['Total'], axis=0)
communes_littorales['decret_2024'] = communes_littorales.merge(df_final[['INSEE_COM', 'decret_2024']], on='INSEE_COM', how='left')['decret_2024']
share_nat_coastlines['decret_2024'] = share_nat_coastlines.apply(lambda x: communes_littorales[communes_littorales['INSEE_COM'] == x['INSEE_COM']]['decret_2024'].values[0], axis=1)
share_nat_coastlines['pprl_ero'] = share_nat_coastlines.apply(lambda x: communes_littorales[communes_littorales['INSEE_COM'] == x['INSEE_COM']]['pprl_ero'].values[0], axis=1)
share_nat_coastlines['pprl_sub'] = share_nat_coastlines.apply(lambda x: communes_littorales[communes_littorales['INSEE_COM'] == x['INSEE_COM']]['pprl_sub'].values[0], axis=1)
# Only select numeric columns for mean aggregation to avoid TypeError
numeric_cols = [
    'Falaise et côte rocheuse supérieure à 20m',
    'Falaise et côte rocheuse inférieure à 20m',
    "Côte d'accumulation sableuse ou sablo-limoneuse",
    "Côte d'accumulation vaseuse",
    'Côte artificialisée',
    'Embouchures',
    'Total'
]
# Create a new column for all combinations of decret_2024, pprl_ero, and pprl_sub
share_nat_coastlines['group'] = (
    'decret_' + share_nat_coastlines['decret_2024'].astype(str) +
    '_ero_' + share_nat_coastlines['pprl_ero'].astype(str) +
    '_sub_' + share_nat_coastlines['pprl_sub'].astype(str)
)

# Plot the mean share for each group (excluding 'Total')
share_nat_coastlines.groupby('group')[numeric_cols[:-1]].mean().plot(
    kind='bar', figsize=(14, 7), title='Share of Nature of Coastlines by Decree and PPRL Status'
)

##### INEC

In [ ]:
evol_litt = gpd.read_file("../data/evol_litt_comm.shp")
evol_litt['decret_2024'] = evol_litt.apply(lambda x: communes_littorales.loc[communes_littorales['INSEE_COM'] == x['INSEE_COM'], 'decret_2024'].values[0], axis=1)
evol_litt = evol_litt.rename({'taux_wo_na': 'inec'}, axis=1)
evol_litt.loc[(evol_litt['inec'] <= -9999) | (evol_litt['inec'] >= 9999), 'inec'] = np.nan
evol_litt['inec'] = evol_litt['inec'].astype(float) * (-1)
evol_litt_muni = evol_litt[['INSEE_COM', 'inec', 'decret_2024', 'amenagemen']].groupby('INSEE_COM').agg({'inec': 'mean', 'decret_2024':'mean', 'amenagemen':'mean'}).reset_index()
evol_litt_muni['geometry'] = evol_litt_muni['INSEE_COM'].apply(lambda x: communes_littorales.loc[communes_littorales['INSEE_COM'] == x, 'geometry'].values[0])

evol_litt_muni = gpd.GeoDataFrame(evol_litt_muni, geometry='geometry', crs=communes_littorales.crs)
# evol_litt_muni['taux_wo_na'] = evol_litt_muni['taux_wo_na'].replace(-9999, 0)
# evol_litt_muni['taux_wo_na'] = evol_litt_muni['taux_wo_na'].replace(9999, 0)
# evol_litt_muni = evol_litt_muni.fillna(0)

In [ ]:
#folium map
# Folium map with improved color mapping: clearer red/green and gray for NaN
import matplotlib.colors as mcolors

def color_inec(val):
    if pd.isna(val):
        return '#cccccc'  # gray for NaN
    elif (val < 0) and (val >= -1):
        return '#7CFC7C'  # light green
    elif (val < -1):
        return '#008000'  # darker green
    elif (val >= 0) and (val <= 1):
        return '#FF7F7F'
    elif (val > 1):
        return '#FF0000'


m = folium.Map(location=[46.5, 2], zoom_start=5, tiles='OpenStreetMap')
folium.GeoJson(
    evol_litt_muni,
    name='Erosion Rate',
    style_function=lambda x: {
        'fillColor': color_inec(x['properties']['inec']),
        'color': 'black',
        'weight': 1,
        'fillOpacity': 0.5
    },
    tooltip=folium.features.GeoJsonTooltip(
        fields=['INSEE_COM', 'inec', 'decret_2024'],
        aliases=['INSEE Code:', 'Erosion Rate (%):', 'Decree 2024:'],
        localize=True
    )
).add_to(m)
m

##### Comparison of all Geo-littoral indicators

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(18, 10))

import seaborn as sns

# Artificialisation rate
sns.violinplot(
    x='decret_2024', y='tx_artif', data=artif_communes, ax=axs[0, 0],
    palette=['lightblue', 'lightcoral'], inner='quart', split=True, cut=0, bw=0.2
)
axs[0, 0].set_title('Coastline artificialisation\nby decree status')
axs[0, 0].set_xlabel('')
axs[0, 0].set_ylabel('Artificialisation rate')
axs[0, 0].set_xticklabels(['Not in decree', 'In decree'])

# PHMA ratio
phma_comm_plot = phma_comm.copy()
phma_comm_plot['ratio'] = phma_comm_plot['ratio'] * 100
sns.violinplot(
    x='decret_2024', y='ratio', data=phma_comm_plot, ax=axs[0, 1],
    palette=['lightblue', 'lightcoral'], inner='quart', split=True, cut=0, bw=0.2
)
axs[0, 1].set_title('Share of the superficy of the\ncommune vulnerable to sea level rise')
axs[0, 1].set_xlabel('')
axs[0, 1].set_ylabel('Share of the exposed superficy (%)')
axs[0, 1].set_xticklabels(['Not in decree', 'In decree'])

# Coastline evolution
sns.violinplot(
    x='decret_2024', y='inec', data=evol_litt_muni, ax=axs[0, 2],
    palette=['lightblue', 'lightcoral'], inner='quart', split=True, cut=0, bw=0.2
)
axs[0, 2].set_title('Coastline evolution by decree status')
axs[0, 2].set_xlabel('')
axs[0, 2].set_ylabel('Coastline evolution rate')
axs[0, 2].set_xticklabels(['Not in decree', 'In decree'])
axs[0, 2].set_ylim(-1, 1)

# Ouvrages erosion
ouvrages_erosion_plot = communes_littorales.merge(
    ouvrages_erosion, on='INSEE_COM', how='left'
)
ouvrages_erosion_plot['nb_ouvrages_erosion'] = ouvrages_erosion_plot['nb_ouvrages_erosion'].fillna(0)
sns.violinplot(
    x='decret_2024', y='nb_ouvrages_erosion', data=ouvrages_erosion_plot, ax=axs[1, 0],
    palette=['lightblue', 'lightcoral'], inner='quart', split=True, cut=0, bw=0.2
)
axs[1, 0].set_title('Number of erosion ouvrages\nby decree status')
axs[1, 0].set_xlabel('')
axs[1, 0].set_ylabel('Number of erosion ouvrages')
axs[1, 0].set_xticklabels(['Not in decree', 'In decree'])

# Ouvrages submersion
ouvrages_submersion_plot = communes_littorales.merge(
    ouvrages_submersion, on='INSEE_COM', how='left'
)
ouvrages_submersion_plot['nb_ouvrages_submersion'] = ouvrages_submersion_plot['nb_ouvrages_submersion'].fillna(0)
sns.violinplot(
    x='decret_2024', y='nb_ouvrages_submersion', data=ouvrages_submersion_plot, ax=axs[1, 1],
    palette=['lightblue', 'lightcoral'], inner='quart', split=True, cut=0, bw=0.2
)
axs[1, 1].set_title('Number of submersion ouvrages\nby decree status')
axs[1, 1].set_xlabel('')
axs[1, 1].set_ylabel('Number of submersion ouvrages')
axs[1, 1].set_xticklabels(['Not in decree', 'In decree'])

# Hide the last subplot if not used
axs[1, 2].axis('off')

plt.tight_layout()


plt.savefig("figures/communes_littorales_analysis_cerema.png", dpi=300, bbox_inches='tight')

##### CatNat

In [ ]:
#two violin plots with the taux of non reconnue for communes in the decree and not in the decree
import seaborn as sns
fig, ax = plt.subplots(1 , 2, figsize=(14, 6))

sns.violinplot(
    x='decret_2024', y='Reconnue', data=df_comm, ax=ax[0],
    palette=['lightblue', 'lightcoral'], inner='quart', split=True, cut=0, bw=0.2
)
ax[0].set_title('Number of CatNat recognized by decree status')
ax[0].set_xlabel('')
ax[0].set_ylabel('Number of CatNat recognized')

sns.violinplot(
    x='decret_2024', y='taux_catnat_recon', data=df_comm, ax=ax[1],
    palette=['lightblue', 'lightcoral'], inner='quart', split=True, cut=0, bw=0.2
)
ax[1].set_title('Rate of CatNat recognized by decree status')
ax[1].set_xlabel('')
ax[1].set_ylabel('Rate of CatNat recognized')

plt.tight_layout()

##### EPCI

In [ ]:
import matplotlib

epci_littorales = communes_littorales.copy()
epci_littorales = epci_littorales[['CODE_EPCI', 'geometry']]
epci_littorales = epci_littorales.dissolve(by='CODE_EPCI', as_index=False)
epci_littorales['decret_2024'] = epci_littorales['CODE_EPCI'].apply(lambda x: communes_littorales[communes_littorales['CODE_EPCI'] == x]['decret_2024'].mean())
epci_littorales['decret_2024'] = np.round(epci_littorales['decret_2024'], 2)  # Round to 2 decimal places

#make a folium map with the EPCI
m_epci = folium.Map(location=[46.5, 2], zoom_start=5, tiles='OpenStreetMap')
def style_function_epci(feature):
    # Use a gradient from light green (for 0) to dark green (for 1)
    value = feature['properties']['decret_2024']
    # Clamp value between 0 and 1
    value = max(0, min(1, value))
    # Use matplotlib colormap for green gradient
    color = matplotlib.colors.to_hex(matplotlib.cm.Greens(value))
    return {
        'fillColor': color,
        'color': color,
        'weight': 1,
        'fillOpacity': 0.7
    }

folium.GeoJson(
    epci_littorales,
    style_function=style_function_epci,
    tooltip=folium.GeoJsonTooltip(fields=['CODE_EPCI', 'decret_2024'], aliases=['EPCI:', 'Share in decree:'])
).add_to(m_epci)

#add title and legend to the map
title_html_epci = """
<div style="position: fixed; 
    top: 10px; left: 50%; transform: translateX(-50%); 
    background-color: white; z-index:9999; 
    font-size:20px; text-align: center; font-weight: bold; 
    padding: 5px; box-shadow: 0px 4px 6px rgba(0, 0, 0, 0.1);">
    Share of communes in the decree on coastal erosion by EPCI
</div>
"""

m_epci.get_root().html.add_child(folium.Element(title_html_epci))

m_epci

##### INSEE Data

In [ ]:
path = '../../Data/INSEE/DS_FILOSOFI_CC_2021_CSV_FR/DS_FILOSOFI_CC_2021_data.csv'
df_filo = pd.read_csv(path, sep=';')
df_filo = df_filo[df_filo.FILOSOFI_MEASURE == 'IR_D9_D1_SL']
df_filo_comm = df_filo[df_filo.GEO_OBJECT == 'COM']
df_filo_epci = df_filo[df_filo.GEO_OBJECT == 'EPCI']
df_filo_comm['CODGEO'] = df_filo_comm['GEO']
df_filo_comm = communes_littorales[['INSEE_COM', 'CODE_EPCI']].merge(df_filo_comm[['CODGEO', 'OBS_VALUE']], left_on='INSEE_COM', right_on='CODGEO', how='left')
print(df_filo_comm.OBS_VALUE.isna().sum(), "nan values in OBS_VALUE")

df_filo_comm.loc[df_filo_comm['CODGEO'].isna(), 'level'] = 'EPCI'
df_filo_comm.loc[df_filo_comm['CODGEO'].notna(), 'level'] = 'COM'
#for the nan value merge with the epci value
df_filo_comm.loc[df_filo_comm['OBS_VALUE'].isna(), 'OBS_VALUE'] = df_filo_comm.CODE_EPCI.map(df_filo_epci.set_index('GEO')['OBS_VALUE'])
df_filo_comm.drop(columns=['CODGEO', 'CODE_EPCI'], inplace=True)

df_insee = df_filo_comm.rename(columns={'OBS_VALUE': 'IR_D9_D1_SL', 'level' : 'level_ir'})

df_filo = pd.read_csv(path, sep=';')
df_filo = df_filo[df_filo.FILOSOFI_MEASURE == 'MED_SL']
df_filo_comm = df_filo[df_filo.GEO_OBJECT == 'COM']
df_filo_epci = df_filo[df_filo.GEO_OBJECT == 'EPCI']
df_filo_comm['CODGEO'] = df_filo_comm['GEO']
df_filo_comm = communes_littorales[['INSEE_COM', 'CODE_EPCI']].merge(df_filo_comm[['CODGEO', 'OBS_VALUE']], left_on='INSEE_COM', right_on='CODGEO', how='left')
print(df_filo_comm.OBS_VALUE.isna().sum(), "nan values in OBS_VALUE")
df_filo_comm.loc[df_filo_comm['CODGEO'].isna(), 'level'] = 'EPCI'
df_filo_comm.loc[df_filo_comm['CODGEO'].notna(), 'level'] = 'COM'
#for the nan value merge with the epci value
df_filo_comm.loc[df_filo_comm['OBS_VALUE'].isna(), 'OBS_VALUE'] = df_filo_comm.CODE_EPCI.map(df_filo_epci.set_index('GEO')['OBS_VALUE'])
df_filo_comm.drop(columns=['CODGEO', 'CODE_EPCI'], inplace=True)

df_insee = df_insee.merge(df_filo_comm.rename(columns={'OBS_VALUE': 'MED_SL', 'level' : 'level_sl'}), on='INSEE_COM', how='left')

In [ ]:
df_test = df_final.merge(df_insee, on='INSEE_COM', how='left')
df_test['decret_pprl_ero'] = df_test['decret_2024'].astype(str) + '_' + df_test['pprl_ero'].astype(str)
df_test = df_test[df_test.Sandy_accumulation > 0.1].copy()
df_test.groupby('decret_pprl_ero')[['inec',
                                     'IR_D9_D1_SL', 'MED_SL']].mean().round(2)


In [ ]:
#plot the mean disposable income and Gini index between decret or not
import seaborn as sns
comm_disp = communes_littorales.merge(rev, left_on='INSEE_COM', right_on='CODGEO', how='left')
fig, ax = plt.subplots(1, 3, figsize=(15, 6))

#replace 's' by nan
comm_disp['Med_Niveau_vie'] = comm_disp['Med_Niveau_vie'].replace('s', np.nan)
comm_disp['Gini'] = comm_disp['Gini'].replace('s', np.nan)
# Replace 'nd' by NaN
comm_disp['Med_Niveau_vie'] = comm_disp['Med_Niveau_vie'].replace('nd', np.nan)
comm_disp['Gini'] = comm_disp['Gini'].replace('nd', np.nan)

comm_disp['Population'] = comm_disp['Population'].replace('s', np.nan)
comm_disp['Population'] = comm_disp['Population'].replace('nd', np.nan)

# Replace commas with dots in 'Med_Niveau_vie' and convert to float
comm_disp['Med_Niveau_vie'] = comm_disp['Med_Niveau_vie'].str.replace(',', '.').astype(float)
# Replace commas with dots in 'Gini' and convert to float
comm_disp['Gini'] = comm_disp['Gini'].str.replace(',', '.').astype(float)
# Replace commas with dots in 'Population' and convert to float
comm_disp['Population'] = comm_disp['Population'].str.replace(',', '.').astype(float)
comm_disp['Population'] = np.log(comm_disp['Population'])  # Log-transform the population for better visualization
# Replace commas with dots in 'Menages' and convert to float

sns.violinplot(x='decret_2024', y='Med_Niveau_vie', data=comm_disp, ax=ax[0], palette=['lightblue', 'lightcoral'])
ax[0].set_title('Median disposable income by commune')
#make horizontal lines for the medians fo each group
ax[0].axhline(y=comm_disp[comm_disp['decret_2024'] == 0]['Med_Niveau_vie'].median(), color='blue', linestyle='--', label='Median non-decret')
ax[0].axhline(y=comm_disp[comm_disp['decret_2024'] == 1]['Med_Niveau_vie'].median(), color='red', linestyle='--', label='Median decret')
ax[0].legend()

sns.violinplot(x='decret_2024', y='Gini', data=comm_disp, ax=ax[1], palette=['lightblue', 'lightcoral'])
ax[1].set_title('Gini index by commune')
#make horizontal lines for the medians fo each group
ax[1].axhline(y=comm_disp[comm_disp['decret_2024'] == 0]['Gini'].median(), color='blue', linestyle='--', label='Median non-decret')
ax[1].axhline(y=comm_disp[comm_disp['decret_2024'] == 1]['Gini'].median(), color='red', linestyle='--', label='Median decret')
ax[1].legend()

sns.violinplot(x='decret_2024', y='Population', data=comm_disp, ax=ax[2], palette=['lightblue', 'lightcoral'])
ax[2].set_ylabel('Log(Population)')
ax[2].set_title('Population by commune')
#make horizontal lines for the medians fo each group
ax[2].axhline(y=comm_disp[comm_disp['decret_2024'] == 0]['Population'].median(), color='blue', linestyle='--', label='Median non-decret')
ax[2].axhline(y=comm_disp[comm_disp['decret_2024'] == 1]['Population'].median(), color='red', linestyle='--', label='Median decret')
ax[2].legend()

#### Voting 

In [ ]:
df_vote_plot = df_vote_final.merge(communes_littorales[['INSEE_COM', 'decret_2024']], on='INSEE_COM', how='left')

# plot the 4 distributions of the share VEC and RN by commune with and without decree
fig, axs = plt.subplots(2,2, figsize=(15,12))
import seaborn as sns

axs = axs.flatten()

sns.violinplot(x='decret_2024', y='share_2024_euro_t1_LVEC_LECO', data=df_vote_plot, ax=axs[0], color='lightgreen')
#make two lines for the medians
axs[0].axhline(df_vote_plot[df_vote_plot['decret_2024'] == 1]['share_2024_euro_t1_LVEC_LECO'].median(), color='green', linestyle='--', label='Median VEC with decree')
axs[0].axhline(df_vote_plot[df_vote_plot['decret_2024'] == 0]['share_2024_euro_t1_LVEC_LECO'].median(), color='green', linestyle=':', label='Median VEC without decree')
axs[0].set_title('Share of green vote by commune with and without decree\neuropean elections 2024')
axs[0].legend()

sns.violinplot(x='decret_2024', y='share_2024_euro_t1_LRN', data=df_vote_plot, ax=axs[1], color='lightcoral')
#make two lines for the medians
axs[1].axhline(df_vote_plot[df_vote_plot['decret_2024'] == 1]['share_2024_euro_t1_LRN'].median(), color='red', linestyle='--', label='Median RN with decree')
axs[1].axhline(df_vote_plot[df_vote_plot['decret_2024'] == 0]['share_2024_euro_t1_LRN'].median(), color='red', linestyle=':', label='Median RN without decree')
axs[1].set_title('Share of RN vote by commune with and without decree\neuropean elections 2024')
axs[1].legend()

axs[2].set_title('Share of green vote by commune with and without decree\neuropean elections 2019')
sns.violinplot(x='decret_2024', y='share_2019_euro_t1_LVEC_LECO', data=df_vote_plot, ax=axs[2], color='lightgreen')
#make two lines for the medians
axs[2].axhline(df_vote_plot[df_vote_plot['decret_2024'] == 1]['share_2019_euro_t1_LVEC_LECO'].median(), color='green', linestyle='--', label='Median VEC with decree')
axs[2].axhline(df_vote_plot[df_vote_plot['decret_2024'] == 0]['share_2019_euro_t1_LVEC_LECO'].median(), color='green', linestyle=':', label='Median VEC without decree')
sns.violinplot(x='decret_2024', y='share_2019_euro_t1_LRN', data=df_vote_plot, ax=axs[3], color='lightcoral')
#make two lines for the medians
axs[3].axhline(df_vote_plot[df_vote_plot['decret_2024'] == 1]['share_2019_euro_t1_LRN'].median(), color='red', linestyle='--', label='Median RN with decree')
axs[3].axhline(df_vote_plot[df_vote_plot['decret_2024'] == 0]['share_2019_euro_t1_LRN'].median(), color='red', linestyle=':', label='Median RN without decree')
axs[3].set_title('Share of RN vote by commune with and without decree\neuropean elections 2019')
for ax in axs:
    ax.set_xlabel('Decree 2024')
    ax.set_ylabel('Share of vote (%)')
    ax.set_xticklabels(['Without decree', 'With decree'])
    ax.grid(True)



#### Exposure 

In [ ]:
communes_plot['relative_snv'] = (communes_plot['nvm_exposed'] - communes_plot['nvm'])/ communes_plot['nvm']
communes_plot.loc[communes_plot['relative_snv'] < -0.9, 'relative_snv'] = np.nan
communes_plot['relative_snv'] = communes_plot['relative_snv'].round(3)*100
communes_plot.loc[communes_plot['relative_snv'].isna(), 'relative_snv'] = np.nan
communes_plot.relative_snv

In [ ]:
import cmocean as cmo 

cmap = cmo.cm.balance
#take 5 colors from the colormap
# colors = [cmap(i) for i in np.linspace(0, 1, 5)]
colors = ['#003049', '#669bbc', '#f0f3bd', '#ed553b', '#c72c41']
#folium map of the relative share of the population exposed to coastal erosion by commune
m_relative = folium.Map(location=[47.0, 2.0], zoom_start=6, tiles='OpenStreetMap')
def style_function_relative(feature):
    taux = feature['properties']['relative_snv']
    # Handle missing or nan values robustly
    if taux == -99:
        color = 'grey'
    elif taux is not None and isinstance(taux, (int, float)) and not np.isnan(taux) and taux < -20:
        color = colors[0]
    elif taux is not None and isinstance(taux, (int, float)) and not np.isnan(taux) and -20 <= taux < -5:
        color = colors[1]
    elif taux is not None and isinstance(taux, (int, float)) and not np.isnan(taux) and -5 <= taux < 5:
        color = colors[2]
    elif taux is not None and isinstance(taux, (int, float)) and not np.isnan(taux) and 5 <= taux < 20:
        color = colors[3]
    elif taux is not None and isinstance(taux, (int, float)) and not np.isnan(taux) and taux >= 20:
        color = colors[4]
    else:
        color = 'grey'
    return {
        'fillColor': color,
        'color': color,
        'weight': 1,
        'fillOpacity': 0.5
    }
folium.GeoJson(
    communes_plot,
    style_function=style_function_relative,
    tooltip=folium.GeoJsonTooltip(fields=['NOM_COM', 'relative_snv'], aliases=['Commune:', 'Relative share of population exposed:'])
).add_to(m_relative)
# Add a legend to the map
import matplotlib.colors as mcolors

# Convert RGBA tuples to hex strings for HTML
color_hex = [mcolors.to_hex(c) for c in colors]

legend_html_relative = f"""
<div style="position: fixed; 
    bottom: 50px; left: 50px; width: 320px; height: 150px; 
    background-color: white; z-index:9999; 
    font-size:14px; padding: 10px;">
    <b>Legend</b><br>
    <i style="display: inline-block; width: 14px; height: 14px; background: {color_hex[0]}; margin-right: 5px; border:1px solid #888;"></i> &lt; <-20% of average<br>
    <i style="display: inline-block; width: 14px; height: 14px; background: {color_hex[1]}; margin-right: 5px; border:1px solid #888;"></i> -20% to -5%<br>
    <i style="display: inline-block; width: 14px; height: 14px; background: {color_hex[2]}; margin-right: 5px; border:1px solid #888;"></i> -5% to 5%<br>
    <i style="display: inline-block; width: 14px; height: 14px; background: {color_hex[3]}; margin-right: 5px; border:1px solid #888;"></i> 5% to 20%<br>
    <i style="display: inline-block; width: 14px; height: 14px; background: {color_hex[4]}; margin-right: 5px; border:1px solid #888;"></i> &gt; >20% of average<br>
</div>
"""
m_relative.get_root().html.add_child(folium.Element(legend_html_relative))
# Add a title to the map
title_html_relative = """
<div style="position: fixed; 
    top: 10px; left: 50%; transform: translateX(-50%); 
    background-color: white; z-index:9999; 
    font-size:20px; text-align: center; font-weight: bold; 
    padding: 5px; box-shadow: 0px 4px 6px rgba(0, 0, 0, 0.1);">
    Niveau de vie of exposed buildings relative to the commune average by commune
</div>
"""
m_relative.get_root().html.add_child(folium.Element(title_html_relative))
# Add a scale bar to the map
m_relative.add_child(folium.LatLngPopup())
# Display the map
m_relative

In [ ]:
communes_plot['relative_exposure_old'] = ((communes_plot['pop_80p_exposed'] + communes_plot['pop_65_79_exposed']) / (communes_plot['pop_80p'] + communes_plot['pop_65_79'])) * communes_plot['pop_2019']/ communes_plot['pop_exposed']
communes_plot['relative_exposure_old'] = communes_plot['relative_exposure_old']
communes_plot.loc[communes_plot['relative_exposure_old'].isna(), 'relative_exposure_old'] = np.nan